In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
simple_returns = pd.read_csv(
    PROCESSED_DIR
    / "simple_returns.csv",
    index_col=0,
    parse_dates=True
)

simple_returns.head()

,SPY,EFA,EEM,AGG,GLD,XLE,XLK
2004-11-19,-0.011117,-0.006474,-0.014297,-0.002044,0.009013,0.011961,-0.015464
2004-11-22,0.004769,0.002933,0.003903,0.001561,0.003796,0.015943,0.001428
2004-11-23,0.001526,0.001949,0.001733,-0.000585,-0.004449,0.006223,-0.004278
2004-11-24,0.002370,0.004539,0.010647,0.000974,0.006704,0.008335,0.008115
2004-11-26,-0.000760,0.005811,0.016606,-0.002920,0.005327,0.004267,-0.001420


In [4]:
print(
    "Start:",
    simple_returns.index.min()
)

print(
    "End:",
    simple_returns.index.max()
)

print(
    "Observations:",
    len(simple_returns)
)

print(
    "Assets:",
    list(simple_returns.columns)
)

print(
    "\nMissing values:"
)

print(
    simple_returns
    .isna()
    .sum()
)

Start: 2004-11-19 00:00:00
End: 2026-10-19 00:00:00
Observations: 5512
Assets: ['SPY', 'EFA', 'EEM', 'AGG', 'GLD', 'XLE', 'XLK']

Missing values:
SPY    0
EFA    0
EEM    0
AGG    0
GLD    0
XLE    0
XLK    0
dtype: int64


In [5]:
shortlist = pd.read_csv(
    TABLE_DIR
    / "portfolio_shortlist.csv"
)

shortlist

,Portfolio Label,N,Subset,Method,Weights
0,Equal Weight Benchmark,7,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",Equal Weight,"[0.1428571429, 0.1428571429, 0.1428571429, 0.1..."
1,Best GMVP,2,"SPY, AGG",GMVP,"[0.0787430197, 0.9212569803]"
2,Best Maximum Sharpe,3,"AGG, GLD, XLK",Maximum Sharpe,"[0.6522812131, 0.1292391967, 0.2184795901]"
3,Best Risk Parity,3,"AGG, GLD, XLK",Risk Parity,"[0.6545410719, 0.1779072155, 0.1675517126]"
4,Best Minimum ES,2,"SPY, AGG",Minimum ES,"[0.0733156417, 0.9266843583]"


In [6]:
def parse_weights(value):

    if isinstance(
        value,
        np.ndarray
    ):
        return value

    if isinstance(
        value,
        list
    ):
        return np.array(
            value,
            dtype=float
        )

    value = (
        str(value)
        .replace("[", "")
        .replace("]", "")
        .replace(",", " ")
    )

    return np.array(
        [
            float(x)
            for x
            in value.split()
        ]
    )

In [7]:
def parse_assets(value):

    if isinstance(
        value,
        list
    ):
        return value

    return [
        x.strip()
        for x
        in str(value).split(",")
    ]

In [8]:
assert len(shortlist) == 5

for _, row in shortlist.iterrows():

    assets = parse_assets(
        row["Subset"]
    )

    weights = parse_weights(
        row["Weights"]
    )

    assert len(assets) == len(weights)

    assert np.isclose(
        weights.sum(),
        1.0,
        atol=1e-6
    )

    assert np.isfinite(
        weights
    ).all()

print(
    "Notebook 06 -> Notebook 07 "
    "portfolio handoff validated."
)

Notebook 06 -> Notebook 07 portfolio handoff validated.


In [9]:
historical_stress_periods = {

    "Global Financial Crisis 2008": (
        "2008-09-01",
        "2009-03-31"
    ),

    "COVID Crash 2020": (
        "2020-02-19",
        "2020-03-23"
    ),

    "Inflation and Rate Shock 2022": (
    "2022-01-03",
    "2022-12-30"
)
}

historical_stress_periods

{'Global Financial Crisis 2008': ('2008-09-01', '2009-03-31'),
 'COVID Crash 2020': ('2020-02-19', '2020-03-23'),
 'Inflation and Rate Shock 2022': ('2022-01-03', '2022-12-30')}

In [10]:
for (
    scenario,
    (
        start_date,
        end_date
    )
) in historical_stress_periods.items():

    stress_data = (
        simple_returns
        .loc[
            start_date:
            end_date
        ]
    )

    print(
        scenario
    )

    print(
        "Start:",
        stress_data.index.min()
    )

    print(
        "End:",
        stress_data.index.max()
    )

    print(
        "Observations:",
        len(stress_data)
    )

    print()

Global Financial Crisis 2008
Start: 2008-09-02 00:00:00
End: 2009-03-31 00:00:00
Observations: 146

COVID Crash 2020
Start: 2020-02-19 00:00:00
End: 2020-03-23 00:00:00
Observations: 24

Inflation and Rate Shock 2022
Start: 2022-01-03 00:00:00
End: 2022-12-30 00:00:00
Observations: 251



## Historical Stress-Test Interpretation

The 2008 Global Financial Crisis and the 2020 COVID crash are contained within
the portfolio development sample used to estimate the frozen candidate
portfolios.

They are therefore interpreted as historical scenario replays rather than
independent out-of-sample validation tests.

The 2022 inflation and rate-hiking episode occurs after the development cutoff
and therefore provides a genuinely out-of-sample stress regime.

The purpose of the historical replay is to measure how the frozen portfolios
would have behaved under known crisis return paths, not to claim predictive
validation.

In [11]:
def portfolio_returns_from_weights(
    assets,
    weights,
    returns
):

    return (
        returns[
            assets
        ]
        @ weights
    )

In [12]:
portfolio_returns = pd.DataFrame(
    index=simple_returns.index
)

for _, row in shortlist.iterrows():

    assets = parse_assets(
        row["Subset"]
    )

    weights = parse_weights(
        row["Weights"]
    )

    portfolio_returns[
        row["Portfolio Label"]
    ] = (
        portfolio_returns_from_weights(
            assets,
            weights,
            simple_returns
        )
    )

portfolio_returns.head()

,Equal Weight Benchmark,Best GMVP,Best Maximum Sharpe,Best Risk Parity,Best Minimum ES
2004-11-19,-0.004060,-0.002758,-0.003547,-0.002326,-0.002709
2004-11-22,0.004905,0.001814,0.001821,0.001937,0.001796
2004-11-23,0.000303,-0.000418,-0.001891,-0.001891,-0.000430
2004-11-24,0.005955,0.001084,0.003275,0.003190,0.001077
2004-11-26,0.003844,-0.002750,-0.001527,-0.001202,-0.002762


In [13]:
print(
    "Portfolio return shape:",
    portfolio_returns.shape
)

print()

print(
    "Missing values:"
)

print(
    portfolio_returns
    .isna()
    .sum()
)

assert (
    portfolio_returns
    .isna()
    .sum()
    .sum()
    == 0
)

Portfolio return shape: (5512, 5)

Missing values:
Equal Weight Benchmark    0
Best GMVP                 0
Best Maximum Sharpe       0
Best Risk Parity          0
Best Minimum ES           0
dtype: int64


In [14]:
def cumulative_return(
    returns
):

    return (
        (
            1
            + returns
        )
        .prod()
        - 1
    )

In [15]:
def maximum_drawdown(
    returns
):

    wealth = (
        1
        + returns
    ).cumprod()

    peak = (
        wealth
        .cummax()
    )

    drawdown = (
        wealth
        / peak
        - 1
    )

    return (
        drawdown
        .min()
    )

In [16]:
def historical_var(
    returns,
    confidence=0.99
):

    return -np.quantile(
        returns,
        1 - confidence
    )

In [17]:
def historical_es(
    returns,
    confidence=0.99
):

    var = historical_var(
        returns,
        confidence
    )

    losses = -returns

    tail_losses = (
        losses[
            losses
            >= var
        ]
    )

    return (
        tail_losses.mean()
    )

In [18]:
stress_rows = []

for (
    scenario,
    (
        start_date,
        end_date
    )
) in historical_stress_periods.items():

    stress_returns = (
        portfolio_returns
        .loc[
            start_date:
            end_date
        ]
    )

    for portfolio in (
        stress_returns.columns
    ):

        r = (
            stress_returns[
                portfolio
            ]
        )

        stress_rows.append({

            "Scenario":
                scenario,

            "Portfolio":
                portfolio,

            "Observations":
                len(r),

            "Cumulative Return":
                cumulative_return(r),

            "Annualized Volatility":
                r.std(ddof=1)
                * np.sqrt(252),

            "Maximum Drawdown":
                maximum_drawdown(r),

            "Worst Daily Return":
                r.min(),

            "Historical VaR 99%":
                historical_var(
                    r,
                    0.99
                ),

            "Historical ES 99%":
                historical_es(
                    r,
                    0.99
                )
        })


historical_stress_results = (
    pd.DataFrame(
        stress_rows
    )
)

historical_stress_results

,Scenario,Portfolio,Observations,Cumulative Return,Annualized Volatility,Maximum Drawdown,Worst Daily Return,Historical VaR 99%,Historical ES 99%
0,Global Financial Crisis 2008,Equal Weight Benchmark,146,-0.240122,0.492433,-0.333831,-0.079559,0.069473,0.074712
1,Global Financial Crisis 2008,Best GMVP,146,0.001582,0.154415,-0.140228,-0.064919,0.021635,0.043748
2,Global Financial Crisis 2008,Best Maximum Sharpe,146,-0.023346,0.168996,-0.136743,-0.055581,0.019850,0.037829
3,Global Financial Crisis 2008,Best Risk Parity,146,-0.001525,0.159120,-0.124503,-0.059034,0.017568,0.038810
4,Global Financial Crisis 2008,Best Minimum ES,146,0.003843,0.154081,-0.139409,-0.065159,0.021551,0.043803
5,COVID Crash 2020,Equal Weight Benchmark,24,-0.279799,0.586909,-0.284521,-0.086847,0.086795,0.086847
6,COVID Crash 2020,Best GMVP,24,-0.041641,0.249284,-0.102475,-0.044394,0.043171,0.044394
7,COVID Crash 2020,Best Maximum Sharpe,24,-0.080935,0.276612,-0.111132,-0.052578,0.048685,0.052578
8,COVID Crash 2020,Best Risk Parity,24,-0.066337,0.251031,-0.104994,-0.049639,0.046223,0.049639
9,COVID Crash 2020,Best Minimum ES,24,-0.039730,0.249199,-0.102009,-0.044092,0.042923,0.044092


### Small-Sample Tail-Risk Caveat

The COVID crash window contains only 24 daily observations.

Historical 99% VaR and Expected Shortfall estimates over such a short window
are therefore highly sample-sensitive and should be interpreted as descriptive
stress statistics rather than stable tail-risk estimates.

The cumulative return and maximum drawdown measures are more reliable for
comparing portfolio behavior across this short crisis episode.

In [19]:
historical_stress_display = (
    historical_stress_results.copy()
)

percentage_columns = [

    "Cumulative Return",
    "Annualized Volatility",
    "Maximum Drawdown",
    "Worst Daily Return",
    "Historical VaR 99%",
    "Historical ES 99%"
]

historical_stress_display[
    percentage_columns
] = (
    historical_stress_display[
        percentage_columns
    ]
    * 100
)

historical_stress_display.round(3)

,Scenario,Portfolio,Observations,Cumulative Return,Annualized Volatility,Maximum Drawdown,Worst Daily Return,Historical VaR 99%,Historical ES 99%
0,Global Financial Crisis 2008,Equal Weight Benchmark,146,-24.012,49.243,-33.383,-7.956,6.947,7.471
1,Global Financial Crisis 2008,Best GMVP,146,0.158,15.442,-14.023,-6.492,2.164,4.375
2,Global Financial Crisis 2008,Best Maximum Sharpe,146,-2.335,16.900,-13.674,-5.558,1.985,3.783
3,Global Financial Crisis 2008,Best Risk Parity,146,-0.153,15.912,-12.450,-5.903,1.757,3.881
4,Global Financial Crisis 2008,Best Minimum ES,146,0.384,15.408,-13.941,-6.516,2.155,4.380
5,COVID Crash 2020,Equal Weight Benchmark,24,-27.980,58.691,-28.452,-8.685,8.680,8.685
6,COVID Crash 2020,Best GMVP,24,-4.164,24.928,-10.248,-4.439,4.317,4.439
7,COVID Crash 2020,Best Maximum Sharpe,24,-8.093,27.661,-11.113,-5.258,4.869,5.258
8,COVID Crash 2020,Best Risk Parity,24,-6.634,25.103,-10.499,-4.964,4.622,4.964
9,COVID Crash 2020,Best Minimum ES,24,-3.973,24.920,-10.201,-4.409,4.292,4.409


In [20]:
stress_cumulative_table = (
    historical_stress_results
    .pivot(
        index="Portfolio",
        columns="Scenario",
        values="Cumulative Return"
    )
)

(
    stress_cumulative_table
    * 100
).round(2)

Scenario,COVID Crash 2020,Global Financial Crisis 2008,Inflation and Rate Shock 2022
Portfolio,,,
Best GMVP,-4.16,0.16,-13.27
Best Maximum Sharpe,-8.09,-2.33,-14.23
Best Minimum ES,-3.97,0.38,-13.26
Best Risk Parity,-6.63,-0.15,-12.97
Equal Weight Benchmark,-27.98,-24.01,-6.32


In [21]:
stress_drawdown_table = (
    historical_stress_results
    .pivot(
        index="Portfolio",
        columns="Scenario",
        values="Maximum Drawdown"
    )
)

(
    stress_drawdown_table
    * 100
).round(2)

Scenario,COVID Crash 2020,Global Financial Crisis 2008,Inflation and Rate Shock 2022
Portfolio,,,
Best GMVP,-10.25,-14.02,-16.44
Best Maximum Sharpe,-11.11,-13.67,-18.10
Best Minimum ES,-10.20,-13.94,-16.41
Best Risk Parity,-10.50,-12.45,-17.11
Equal Weight Benchmark,-28.45,-33.38,-17.37


In [22]:
stress_es_table = (
    historical_stress_results
    .pivot(
        index="Portfolio",
        columns="Scenario",
        values="Historical ES 99%"
    )
)

(
    stress_es_table
    * 100
).round(3)

Scenario,COVID Crash 2020,Global Financial Crisis 2008,Inflation and Rate Shock 2022
Portfolio,,,
Best GMVP,4.439,4.375,1.421
Best Maximum Sharpe,5.258,3.783,1.940
Best Minimum ES,4.409,4.380,1.413
Best Risk Parity,4.964,3.881,1.765
Equal Weight Benchmark,8.685,7.471,3.207


In [23]:
for (
    scenario,
    (
        start_date,
        end_date
    )
) in historical_stress_periods.items():

    stress_returns = (
        portfolio_returns
        .loc[
            start_date:
            end_date
        ]
    )

    growth = (
        1
        + stress_returns
    ).cumprod()

    plt.figure(
        figsize=(11, 6)
    )

    for portfolio in (
        growth.columns
    ):

        plt.plot(
            growth.index,
            growth[
                portfolio
            ],
            label=portfolio
        )

    plt.title(
        f"Portfolio Performance During {scenario}"
    )

    plt.xlabel(
        "Date"
    )

    plt.ylabel(
        "Growth of $1"
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend(
        bbox_to_anchor=(
            1.02,
            1
        ),
        loc="upper left"
    )

    plt.tight_layout()

    filename = (
        scenario
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    plt.savefig(
        FIGURE_DIR
        / f"stress_{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_27636\1533024713.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Hypothetical Scenario Assumptions

The hypothetical stress scenarios are deterministic shock assumptions designed
to test portfolio sensitivity under plausible but severe market conditions.

They are not forecasts and do not represent estimated probabilities.

Each asset shock is applied directly to the frozen portfolio weights, allowing
the analysis to compare how different portfolio structures respond to the same
market event.

In [24]:
hypothetical_shocks = {
    "Severe Equity Crash": {
        "SPY": -0.30,
        "EFA": -0.35,
        "EEM": -0.40,
        "AGG": 0.05,
        "GLD": 0.10,
        "XLE": -0.30,
        "XLK": -0.35
    },

    "Inflation and Rate Shock": {
        "SPY": -0.15,
        "EFA": -0.18,
        "EEM": -0.20,
        "AGG": -0.12,
        "GLD": 0.08,
        "XLE": 0.10,
        "XLK": -0.22
    },

    "Technology Sector Crash": {
        "SPY": -0.20,
        "EFA": -0.12,
        "EEM": -0.15,
        "AGG": 0.03,
        "GLD": 0.06,
        "XLE": -0.08,
        "XLK": -0.40
    },

    "Emerging Market Crisis": {
        "SPY": -0.12,
        "EFA": -0.20,
        "EEM": -0.40,
        "AGG": 0.04,
        "GLD": 0.08,
        "XLE": -0.18,
        "XLK": -0.15
    }
}

hypothetical_shocks

{'Severe Equity Crash': {'SPY': -0.3,
  'EFA': -0.35,
  'EEM': -0.4,
  'AGG': 0.05,
  'GLD': 0.1,
  'XLE': -0.3,
  'XLK': -0.35},
 'Inflation and Rate Shock': {'SPY': -0.15,
  'EFA': -0.18,
  'EEM': -0.2,
  'AGG': -0.12,
  'GLD': 0.08,
  'XLE': 0.1,
  'XLK': -0.22},
 'Technology Sector Crash': {'SPY': -0.2,
  'EFA': -0.12,
  'EEM': -0.15,
  'AGG': 0.03,
  'GLD': 0.06,
  'XLE': -0.08,
  'XLK': -0.4},
 'Emerging Market Crisis': {'SPY': -0.12,
  'EFA': -0.2,
  'EEM': -0.4,
  'AGG': 0.04,
  'GLD': 0.08,
  'XLE': -0.18,
  'XLK': -0.15}}

In [25]:
shock_assumption_table = (
    pd.DataFrame(
        hypothetical_shocks
    )
    .T
    * 100
)

shock_assumption_table.round(2)

,SPY,EFA,EEM,AGG,GLD,XLE,XLK
Severe Equity Crash,-30.0,-35.0,-40.0,5.0,10.0,-30.0,-35.0
Inflation and Rate Shock,-15.0,-18.0,-20.0,-12.0,8.0,10.0,-22.0
Technology Sector Crash,-20.0,-12.0,-15.0,3.0,6.0,-8.0,-40.0
Emerging Market Crisis,-12.0,-20.0,-40.0,4.0,8.0,-18.0,-15.0


In [26]:
hypothetical_rows = []

for scenario, shock_dict in hypothetical_shocks.items():

    asset_shocks = pd.Series(shock_dict)

    for _, row in shortlist.iterrows():

        assets = [
            x.strip()
            for x in row["Subset"].split(",")
        ]

        weights = parse_weights(
            row["Weights"]
        )

        portfolio_shock = np.dot(
            weights,
            asset_shocks[assets].values
        )

        hypothetical_rows.append({
            "Scenario": scenario,
            "Portfolio": row["Portfolio Label"],
            "Portfolio Shock Return": portfolio_shock
        })


hypothetical_results = pd.DataFrame(
    hypothetical_rows
)

hypothetical_results

,Scenario,Portfolio,Portfolio Shock Return
0,Severe Equity Crash,Equal Weight Benchmark,-0.221429
1,Severe Equity Crash,Best GMVP,0.022440
2,Severe Equity Crash,Best Maximum Sharpe,-0.030930
3,Severe Equity Crash,Best Risk Parity,-0.008125
4,Severe Equity Crash,Best Minimum ES,0.024340
5,Inflation and Rate Shock,Equal Weight Benchmark,-0.098571
6,Inflation and Rate Shock,Best GMVP,-0.122362
7,Inflation and Rate Shock,Best Maximum Sharpe,-0.116000
8,Inflation and Rate Shock,Best Risk Parity,-0.101174
9,Inflation and Rate Shock,Best Minimum ES,-0.122199


In [27]:
print(
    "Hypothetical scenario rows:",
    len(hypothetical_results)
)

assert (
    len(hypothetical_results)
    == len(hypothetical_shocks)
       * len(shortlist)
)

Hypothetical scenario rows: 20


In [28]:
hypothetical_display = (
    hypothetical_results.copy()
)

hypothetical_display[
    "Portfolio Shock Return"
] *= 100

hypothetical_display[
    "Portfolio Shock Return"
] = hypothetical_display[
    "Portfolio Shock Return"
].round(2)

hypothetical_display

,Scenario,Portfolio,Portfolio Shock Return
0,Severe Equity Crash,Equal Weight Benchmark,-22.14
1,Severe Equity Crash,Best GMVP,2.24
2,Severe Equity Crash,Best Maximum Sharpe,-3.09
3,Severe Equity Crash,Best Risk Parity,-0.81
4,Severe Equity Crash,Best Minimum ES,2.43
5,Inflation and Rate Shock,Equal Weight Benchmark,-9.86
6,Inflation and Rate Shock,Best GMVP,-12.24
7,Inflation and Rate Shock,Best Maximum Sharpe,-11.60
8,Inflation and Rate Shock,Best Risk Parity,-10.12
9,Inflation and Rate Shock,Best Minimum ES,-12.22


In [29]:
hypothetical_table = (
    hypothetical_results
    .pivot(
        index="Portfolio",
        columns="Scenario",
        values="Portfolio Shock Return"
    )
)

(
    hypothetical_table
    * 100
).round(2)

Scenario,Emerging Market Crisis,Inflation and Rate Shock,Severe Equity Crash,Technology Sector Crash
Portfolio,,,,
Best GMVP,2.74,-12.24,2.24,1.19
Best Maximum Sharpe,0.37,-11.60,-3.09,-6.01
Best Minimum ES,2.83,-12.22,2.43,1.31
Best Risk Parity,1.53,-10.12,-0.81,-3.67
Equal Weight Benchmark,-13.29,-9.86,-22.14,-12.29


In [30]:
hypothetical_plot = (
    hypothetical_table
    * 100
)

hypothetical_plot.T.plot(
    kind="bar",
    figsize=(12, 7)
)

plt.axhline(
    0,
    linewidth=1
)

plt.ylabel(
    "Portfolio Return Under Shock (%)"
)

plt.xlabel(
    "Hypothetical Scenario"
)

plt.title(
    "Portfolio Performance Under Hypothetical Stress Scenarios"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "hypothetical_stress_portfolios.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_27636\1953701834.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
worst_hypothetical = pd.DataFrame({
    "Worst Scenario":
        hypothetical_table.idxmin(
            axis=1
        ),

    "Worst Scenario Return":
        hypothetical_table.min(
            axis=1
        )
})

worst_hypothetical[
    "Worst Scenario Return"
] *= 100

worst_hypothetical.round(2)

,Worst Scenario,Worst Scenario Return
Portfolio,,
Best GMVP,Inflation and Rate Shock,-12.24
Best Maximum Sharpe,Inflation and Rate Shock,-11.60
Best Minimum ES,Inflation and Rate Shock,-12.22
Best Risk Parity,Inflation and Rate Shock,-10.12
Equal Weight Benchmark,Severe Equity Crash,-22.14


In [32]:
hypothetical_summary = pd.DataFrame({
    "Average Shock Return":
        hypothetical_table.mean(
            axis=1
        ),

    "Worst Shock Return":
        hypothetical_table.min(
            axis=1
        ),

    "Best Shock Return":
        hypothetical_table.max(
            axis=1
        )
})

(
    hypothetical_summary
    * 100
).round(2)

,Average Shock Return,Worst Shock Return,Best Shock Return
Portfolio,,,
Best GMVP,-1.52,-12.24,2.74
Best Maximum Sharpe,-5.08,-11.60,0.37
Best Minimum ES,-1.41,-12.22,2.83
Best Risk Parity,-3.27,-10.12,1.53
Equal Weight Benchmark,-14.39,-22.14,-9.86


In [33]:
historical_resilience = (
    historical_stress_results
    .groupby("Portfolio")
    .agg({
        "Cumulative Return": "mean",
        "Maximum Drawdown": "mean",
        "Historical ES 99%": "mean"
    })
)

historical_resilience

,Cumulative Return,Maximum Drawdown,Historical ES 99%
Portfolio,,,
Best GMVP,-0.057601,-0.135696,0.034118
Best Maximum Sharpe,-0.082207,-0.142961,0.036602
Best Minimum ES,-0.056149,-0.135173,0.034009
Best Risk Parity,-0.065855,-0.133531,0.035368
Equal Weight Benchmark,-0.194365,-0.264009,0.064541


In [34]:
historical_worst_case = (
    historical_stress_results
    .groupby("Portfolio")
    .agg({
        "Cumulative Return": "min",
        "Maximum Drawdown": "min",
        "Historical ES 99%": "max"
    })
)

historical_worst_case.columns = [
    "Worst Historical Cumulative Return",
    "Worst Historical Drawdown",
    "Worst Historical ES 99%"
]

historical_worst_case

,Worst Historical Cumulative Return,Worst Historical Drawdown,Worst Historical ES 99%
Portfolio,,,
Best GMVP,-0.132744,-0.164387,0.044394
Best Maximum Sharpe,-0.142339,-0.181008,0.052578
Best Minimum ES,-0.132561,-0.164101,0.044092
Best Risk Parity,-0.129702,-0.171095,0.049639
Equal Weight Benchmark,-0.279799,-0.333831,0.086847


In [35]:
historical_worst_display = (
    historical_worst_case
    * 100
)

historical_worst_display.round(2)

,Worst Historical Cumulative Return,Worst Historical Drawdown,Worst Historical ES 99%
Portfolio,,,
Best GMVP,-13.27,-16.44,4.44
Best Maximum Sharpe,-14.23,-18.10,5.26
Best Minimum ES,-13.26,-16.41,4.41
Best Risk Parity,-12.97,-17.11,4.96
Equal Weight Benchmark,-27.98,-33.38,8.68


In [36]:
stress_rankings = pd.DataFrame(
    index=historical_worst_case.index
)

stress_rankings[
    "Historical Return Rank"
] = (
    historical_worst_case[
        "Worst Historical Cumulative Return"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

stress_rankings[
    "Historical Drawdown Rank"
] = (
    historical_worst_case[
        "Worst Historical Drawdown"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

stress_rankings[
    "Historical ES Rank"
] = (
    historical_worst_case[
        "Worst Historical ES 99%"
    ]
    .rank(
        ascending=True,
        method="min"
    )
)

stress_rankings[
    "Hypothetical Worst-Shock Rank"
] = (
    hypothetical_summary[
        "Worst Shock Return"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

stress_rankings

,Historical Return Rank,Historical Drawdown Rank,Historical ES Rank,Hypothetical Worst-Shock Rank
Portfolio,,,,
Best GMVP,3.0,2.0,2.0,4.0
Best Maximum Sharpe,4.0,4.0,4.0,2.0
Best Minimum ES,2.0,1.0,1.0,3.0
Best Risk Parity,1.0,3.0,3.0,1.0
Equal Weight Benchmark,5.0,5.0,5.0,5.0


In [37]:
stress_rankings[
    "Average Stress Rank"
] = (
    stress_rankings.mean(
        axis=1
    )
)

stress_rankings[
    "Overall Stress Rank"
] = (
    stress_rankings[
        "Average Stress Rank"
    ]
    .rank(
        ascending=True,
        method="min"
    )
)

stress_rankings.sort_values(
    "Overall Stress Rank"
)

,Historical Return Rank,Historical Drawdown Rank,Historical ES Rank,Hypothetical Worst-Shock Rank,Average Stress Rank,Overall Stress Rank
Portfolio,,,,,,
Best Minimum ES,2.0,1.0,1.0,3.0,1.75,1.0
Best Risk Parity,1.0,3.0,3.0,1.0,2.00,2.0
Best GMVP,3.0,2.0,2.0,4.0,2.75,3.0
Best Maximum Sharpe,4.0,4.0,4.0,2.0,3.50,4.0
Equal Weight Benchmark,5.0,5.0,5.0,5.0,5.00,5.0


In [38]:
stress_decision_table = pd.concat(
    [
        historical_worst_case,
        hypothetical_summary[
            [
                "Worst Shock Return"
            ]
        ],
        stress_rankings[
            [
                "Overall Stress Rank"
            ]
        ]
    ],
    axis=1
)

stress_decision_table

,Worst Historical Cumulative Return,Worst Historical Drawdown,Worst Historical ES 99%,Worst Shock Return,Overall Stress Rank
Portfolio,,,,,
Best GMVP,-0.132744,-0.164387,0.044394,-0.122362,3.0
Best Maximum Sharpe,-0.142339,-0.181008,0.052578,-0.116000,4.0
Best Minimum ES,-0.132561,-0.164101,0.044092,-0.122199,1.0
Best Risk Parity,-0.129702,-0.171095,0.049639,-0.101174,2.0
Equal Weight Benchmark,-0.279799,-0.333831,0.086847,-0.221429,5.0


In [39]:
stress_scorecard = pd.concat(
    [
        historical_worst_case[
            [
                "Worst Historical Cumulative Return",
                "Worst Historical Drawdown",
                "Worst Historical ES 99%"
            ]
        ],
        hypothetical_summary[
            [
                "Average Shock Return",
                "Worst Shock Return"
            ]
        ],
        stress_rankings[
            [
                "Overall Stress Rank"
            ]
        ]
    ],
    axis=1
)

stress_scorecard

,Worst Historical Cumulative Return,Worst Historical Drawdown,Worst Historical ES 99%,Average Shock Return,Worst Shock Return,Overall Stress Rank
Portfolio,,,,,,
Best GMVP,-0.132744,-0.164387,0.044394,-0.015158,-0.122362,3.0
Best Maximum Sharpe,-0.142339,-0.181008,0.052578,-0.050835,-0.116000,4.0
Best Minimum ES,-0.132561,-0.164101,0.044092,-0.014113,-0.122199,1.0
Best Risk Parity,-0.129702,-0.171095,0.049639,-0.032682,-0.101174,2.0
Equal Weight Benchmark,-0.279799,-0.333831,0.086847,-0.143929,-0.221429,5.0


In [40]:
stress_scorecard_display = (
    stress_scorecard.copy()
)

percentage_columns = [
    "Worst Historical Cumulative Return",
    "Worst Historical Drawdown",
    "Worst Historical ES 99%",
    "Average Shock Return",
    "Worst Shock Return"
]

stress_scorecard_display[
    percentage_columns
] *= 100

stress_scorecard_display.round(2)

,Worst Historical Cumulative Return,Worst Historical Drawdown,Worst Historical ES 99%,Average Shock Return,Worst Shock Return,Overall Stress Rank
Portfolio,,,,,,
Best GMVP,-13.27,-16.44,4.44,-1.52,-12.24,3.0
Best Maximum Sharpe,-14.23,-18.10,5.26,-5.08,-11.60,4.0
Best Minimum ES,-13.26,-16.41,4.41,-1.41,-12.22,1.0
Best Risk Parity,-12.97,-17.11,4.96,-3.27,-10.12,2.0
Equal Weight Benchmark,-27.98,-33.38,8.68,-14.39,-22.14,5.0


In [41]:
benchmark_name = (
    "Equal Weight Benchmark"
)

benchmark_drawdown = abs(
    stress_scorecard.loc[
        benchmark_name,
        "Worst Historical Drawdown"
    ]
)

benchmark_es = (
    stress_scorecard.loc[
        benchmark_name,
        "Worst Historical ES 99%"
    ]
)

benchmark_worst_shock = abs(
    stress_scorecard.loc[
        benchmark_name,
        "Worst Shock Return"
    ]
)

protection_table = pd.DataFrame(
    index=stress_scorecard.index
)

protection_table[
    "Drawdown Reduction vs Equal Weight"
] = (
    1
    -
    abs(
        stress_scorecard[
            "Worst Historical Drawdown"
        ]
    )
    / benchmark_drawdown
)

protection_table[
    "ES Reduction vs Equal Weight"
] = (
    1
    -
    stress_scorecard[
        "Worst Historical ES 99%"
    ]
    / benchmark_es
)

protection_table[
    "Worst-Shock Reduction vs Equal Weight"
] = (
    1
    -
    abs(
        stress_scorecard[
            "Worst Shock Return"
        ]
    )
    / benchmark_worst_shock
)

protection_table

,Drawdown Reduction vs Equal Weight,ES Reduction vs Equal Weight,Worst-Shock Reduction vs Equal Weight
Portfolio,,,
Best GMVP,0.507576,0.488826,0.447396
Best Maximum Sharpe,0.457784,0.394591,0.476128
Best Minimum ES,0.508431,0.492305,0.448131
Best Risk Parity,0.487481,0.428434,0.543086
Equal Weight Benchmark,0.000000,0.000000,0.000000


### Stress-Ranking Method

The overall stress ranking assigns equal importance to four dimensions:

1. worst historical cumulative return;
2. worst historical drawdown;
3. worst historical 99% Expected Shortfall;
4. worst hypothetical scenario return.

The equal-weight ranking is a transparent decision aid rather than a formal
optimization objective.

The final portfolio recommendation therefore considers these ranks together
with out-of-sample performance, concentration and risk-budgeting evidence in
Notebook 08.

In [42]:
(
    protection_table
    * 100
).round(2)

,Drawdown Reduction vs Equal Weight,ES Reduction vs Equal Weight,Worst-Shock Reduction vs Equal Weight
Portfolio,,,
Best GMVP,50.76,48.88,44.74
Best Maximum Sharpe,45.78,39.46,47.61
Best Minimum ES,50.84,49.23,44.81
Best Risk Parity,48.75,42.84,54.31
Equal Weight Benchmark,0.00,0.00,0.00


In [43]:
comparison_plot = pd.DataFrame(
    {
        "Worst Historical Return":
            stress_scorecard[
                "Worst Historical Cumulative Return"
            ]
            * 100,

        "Worst Hypothetical Shock":
            stress_scorecard[
                "Worst Shock Return"
            ]
            * 100
    }
)

comparison_plot.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.axhline(
    0,
    linewidth=1
)

plt.ylabel(
    "Portfolio Return (%)"
)

plt.title(
    "Historical and Hypothetical "
    "Stress Vulnerability"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "stress_resilience_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_27636\1708046708.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [44]:
historical_scenario_winners = (
    stress_cumulative_table
    .idxmax(
        axis=0
    )
    .to_frame(
        name="Highest Cumulative Return Portfolio"
    )
)

historical_scenario_winners

,Highest Cumulative Return Portfolio
Scenario,
COVID Crash 2020,Best Minimum ES
Global Financial Crisis 2008,Best Minimum ES
Inflation and Rate Shock 2022,Equal Weight Benchmark


In [45]:
hypothetical_scenario_winners = (
    hypothetical_table
    .idxmax(
        axis=0
    )
    .to_frame(
        name="Highest Scenario Return Portfolio"
    )
)

hypothetical_scenario_winners

,Highest Scenario Return Portfolio
Scenario,
Emerging Market Crisis,Best Minimum ES
Inflation and Rate Shock,Equal Weight Benchmark
Severe Equity Crash,Best Minimum ES
Technology Sector Crash,Best Minimum ES


In [46]:
portfolio_vulnerability = pd.DataFrame(
    {
        "Historical Worst Case":
            stress_cumulative_table
            .idxmin(axis=1),

        "Hypothetical Worst Case":
            hypothetical_table
            .idxmin(axis=1)
    }
)

portfolio_vulnerability

,Historical Worst Case,Hypothetical Worst Case
Portfolio,,
Best GMVP,Inflation and Rate Shock 2022,Inflation and Rate Shock
Best Maximum Sharpe,Inflation and Rate Shock 2022,Inflation and Rate Shock
Best Minimum ES,Inflation and Rate Shock 2022,Inflation and Rate Shock
Best Risk Parity,Inflation and Rate Shock 2022,Inflation and Rate Shock
Equal Weight Benchmark,COVID Crash 2020,Severe Equity Crash


In [47]:
concentration_rows = []

for _, row in shortlist.iterrows():

    assets = [
        asset.strip()
        for asset in row["Subset"].split(",")
    ]

    weights = parse_weights(
        row["Weights"]
    )

    concentration_rows.append(
        {
            "Portfolio":
                row["Portfolio Label"],

            "Number of Assets":
                len(assets),

            "Largest Weight":
                weights.max(),

            "Largest Holding":
                assets[
                    np.argmax(weights)
                ]
        }
    )

concentration_summary = (
    pd.DataFrame(
        concentration_rows
    )
    .set_index(
        "Portfolio"
    )
)

concentration_summary[
    "Largest Weight"
] *= 100

concentration_summary.round(2)

,Number of Assets,Largest Weight,Largest Holding
Portfolio,,,
Equal Weight Benchmark,7,14.29,SPY
Best GMVP,2,92.13,AGG
Best Maximum Sharpe,3,65.23,AGG
Best Risk Parity,3,65.45,AGG
Best Minimum ES,2,92.67,AGG


In [48]:
final_stress_decision = pd.concat(
    [
        stress_scorecard[
            [
                "Overall Stress Rank",
                
            ]
        ],

        concentration_summary[
            [
                "Number of Assets",
                "Largest Holding",
                "Largest Weight"
            ]
        ]
    ],
    axis=1
)

final_stress_decision

,Overall Stress Rank,Number of Assets,Largest Holding,Largest Weight
Portfolio,,,,
Best GMVP,3.0,2,AGG,92.125698
Best Maximum Sharpe,4.0,3,AGG,65.228121
Best Minimum ES,1.0,2,AGG,92.668436
Best Risk Parity,2.0,3,AGG,65.454107
Equal Weight Benchmark,5.0,7,SPY,14.285714


In [49]:
historical_stress_results.to_csv(
    TABLE_DIR
    / "historical_stress_results.csv",
    index=False
)

stress_cumulative_table.to_csv(
    TABLE_DIR
    / "historical_stress_cumulative_returns.csv"
)

stress_drawdown_table.to_csv(
    TABLE_DIR
    / "historical_stress_drawdowns.csv"
)

stress_es_table.to_csv(
    TABLE_DIR
    / "historical_stress_es99.csv"
)

hypothetical_results.to_csv(
    TABLE_DIR
    / "hypothetical_stress_results.csv",
    index=False
)

stress_rankings.to_csv(
    TABLE_DIR
    / "stress_resilience_rankings.csv"
)

stress_scorecard.to_csv(
    TABLE_DIR
    / "stress_resilience_scorecard.csv"
)

protection_table.to_csv(
    TABLE_DIR
    / "stress_protection_vs_equal_weight.csv"
)

concentration_summary.to_csv(
    TABLE_DIR
    / "portfolio_concentration_summary.csv"
)

final_stress_decision.to_csv(
    TABLE_DIR
    / "final_stress_decision_table.csv"
)

print(
    "Notebook 07 tables saved successfully."
)

Notebook 07 tables saved successfully.


In [50]:
shock_assumption_table.to_csv(
    TABLE_DIR
    / "hypothetical_shock_assumptions.csv"
)

In [51]:
assert simple_returns.shape == (
    5512,
    7
)

assert (
    simple_returns.index.max()
    == pd.Timestamp(
        "2026-10-19"
    )
)

assert len(shortlist) == 5

assert (
    portfolio_returns.shape
    == (
        5512,
        5
    )
)

assert (
    portfolio_returns
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    len(
        historical_stress_results
    )
    == 15
)

assert (
    len(
        hypothetical_results
    )
    == 20
)

assert (
    hypothetical_table.shape
    == (
        5,
        4
    )
)

assert (
    historical_worst_case.shape[0]
    == 5
)

assert (
    stress_rankings.shape[0]
    == 5
)

assert (
    hypothetical_results[
        "Portfolio Shock Return"
    ]
    .notna()
    .all()
)

assert (
    final_stress_decision[
        "Overall Stress Rank"
    ]
    .min()
    == 1
)

for _, row in shortlist.iterrows():

    weights = parse_weights(
        row["Weights"]
    )

    assert np.isclose(
        weights.sum(),
        1.0,
        atol=1e-6
    )

print(
    "All Notebook 07 final "
    "validation checks passed."
)

All Notebook 07 final validation checks passed.
